## Setup
Run notebooks 01–07 first. Install dependencies once from the project root with `python -m pip install -r requirements.txt`.

In [1]:
from pathlib import Path
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate/"data").is_dir() and (candidate/"notebooks").is_dir(): PROJECT_ROOT=candidate; break
else: raise RuntimeError("Start Jupyter inside the project so its root can be found.")
DATA_DIR=PROJECT_ROOT/"data"; INTERMEDIATE_DIR=DATA_DIR/"02_intermediate"; MODEL_DIR=PROJECT_ROOT/"models"; REPORT_DIR=PROJECT_ROOT/"reports"
INTERMEDIATE_DIR.mkdir(parents=True,exist_ok=True); MODEL_DIR.mkdir(parents=True,exist_ok=True); REPORT_DIR.mkdir(parents=True,exist_ok=True)
print(f"Project root: {PROJECT_ROOT}")

Project root: c:\Users\acer\Desktop\Programs\Projects_for _data_science\Fraud_Detection_Model


# 11 · Monitoring, drift, and feedback

Monitoring checks whether data, score distributions, and outcomes remain useful after launch. This notebook illustrates period comparisons using the historical validation window; it is not a production feed and drift alone does not prove the model is failing.

In [2]:
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, precision_score, recall_score

train=pd.read_pickle(INTERMEDIATE_DIR/"train_features.pkl"); artifact=joblib.load(MODEL_DIR/"best_experiment_model.joblib")
features=list(artifact["feature_cols"]); ordered=train.sort_values("TransactionDT",kind="stable").reset_index(drop=True)
val=ordered.iloc[int(len(ordered)*0.8):].copy(); cut=len(val)//2
periods={"validation_early":val.iloc[:cut].copy(),"validation_late":val.iloc[cut:].copy()}
for frame in periods.values():
    X=frame[features].replace([np.inf,-np.inf],np.nan)
    if artifact.get("imputer") is not None: X=artifact["imputer"].transform(X)
    if artifact.get("scaler") is not None: X=artifact["scaler"].transform(X)
    frame["score"]=artifact["model"].predict_proba(X)[:,1]
print({name:(len(f),f.TransactionDT.min(),f.TransactionDT.max()) for name,f in periods.items()})

{'validation_early': (59054, np.int64(12192900), np.int64(13990904)), 'validation_late': (59054, np.int64(13990941), np.int64(15811131))}


## 1. Data quality and score movement

In [3]:
def data_quality(frame):
    X=frame[features].replace([np.inf,-np.inf],np.nan)
    return pd.Series({"rows":len(frame),"missing_feature_cells_pct":X.isna().to_numpy().mean(),"fraud_rate":frame.isFraud.mean(),"mean_score":frame.score.mean(),"p95_score":frame.score.quantile(.95)})
quality=pd.DataFrame({name:data_quality(frame) for name,frame in periods.items()}).T; quality

,rows,missing_feature_cells_pct,fraud_rate,mean_score,p95_score
validation_early,59054.0,0.436150,0.031344,0.221283,0.694783
validation_late,59054.0,0.422597,0.037474,0.227064,0.728001


## 2. Population Stability Index

PSI summarizes change in a binned distribution. Bins are fixed from the earlier window, with smoothing for empty bins. Treat it as a screening signal, not a significance test or an automatic retraining trigger.

In [4]:
def psi(reference,current,bins=10,epsilon=1e-6):
    ref=pd.Series(reference).dropna().to_numpy(); cur=pd.Series(current).dropna().to_numpy()
    if not len(ref) or not len(cur): return np.nan
    edges=np.unique(np.quantile(ref,np.linspace(0,1,bins+1)))
    if len(edges)<3: return 0.0
    edges[0],edges[-1]=-np.inf,np.inf
    a=np.histogram(ref,bins=edges)[0]/len(ref); b=np.histogram(cur,bins=edges)[0]/len(cur)
    a=np.clip(a,epsilon,None); b=np.clip(b,epsilon,None)
    return float(np.sum((b-a)*np.log(b/a)))
ref,cur=periods["validation_early"],periods["validation_late"]
cols=[c for c in ["TransactionAmt","card1_txn_count_1h","n_missing"] if c in ref]
drift=pd.DataFrame([{"variable":"score","psi":psi(ref.score,cur.score)}]+[{"variable":c,"psi":psi(ref[c],cur[c])} for c in cols])
drift.sort_values("psi",ascending=False)

,variable,psi
3,n_missing,0.030587
0,score,0.004592
1,TransactionAmt,0.002346
2,card1_txn_count_1h,0.001118


## 3. Outcome metrics after labels mature

In [5]:
MONITOR_THRESHOLD=0.53  # illustrative; align with validated policy
rows=[]
for name,frame in periods.items():
    truth=frame.isFraud.astype(int); pred=(frame.score>=MONITOR_THRESHOLD).astype(int)
    rows.append({"period":name,"rows":len(frame),"fraud_rate":truth.mean(),"average_precision":average_precision_score(truth,frame.score),"precision_at_threshold":precision_score(truth,pred,zero_division=0),"recall_at_threshold":recall_score(truth,pred,zero_division=0),"flag_rate":pred.mean()})
pd.DataFrame(rows).set_index("period")

,rows,fraud_rate,average_precision,precision_at_threshold,recall_at_threshold,flag_rate
period,,,,,,
validation_early,59054,0.031344,0.490262,0.214346,0.690978,0.101043
validation_late,59054,0.037474,0.527015,0.241071,0.707637,0.110001


## 4. Monitoring playbook

Track input schema and missingness, feature freshness, prediction latency/failures, score and action rates, label maturity, fraud capture, false declines, review queue volume, and customer outcomes. Assign owners and alert thresholds based on historical variation. Investigate pipeline and population changes before retraining. Preserve prediction-time features and model versions so delayed labels can be joined without leakage.